# Swiss Population Data Cleaning

This notebook loads and cleans population data by postal code from the
Swiss Federal Statistical Office.

The latest available data, for 2025, is used in this analysis.

This file does:
- Loading the BFS population Excel file
- Understanding its rows and columns
- Selecting the most recent year
- Selecting total population without demographic double-counting
- Cleaning postal codes
- Saving a clean population CSV

In [1]:
# First download the data from: https://www.bfs.admin.ch/bfs/en/home/statistics/catalogues-databases.assetdetail.36681794.html
# 'su-e-01.02.03.07.xlsx' - store it inside the data folder

In [2]:
import pandas as pd

In [3]:
# load the data
# use - conda install openpyxl -y
population_file = "data/su-e-01.02.03.07.xlsx"

excel_file = pd.ExcelFile(population_file)

# The workbook contains one sheet for every year from 2010 to 2025.
print(excel_file.sheet_names)

['2025', '2024', '2023', '2022', '2021', '2020', '2019', '2018', '2017', '2016', '2015', '2014', '2013', '2012', '2011', '2010']


In [4]:
# using 2025 because it is the most recent year.
population_preview = pd.read_excel(
    population_file,
    sheet_name="2025",
    header=None,
    nrows=10
)

population_preview

,0,1,2,3,4,5,6,7,8,9,...,22,23,24,25,26,27,28,29,30,31
0,su-e-01.02.03.07,"Permanent resident population by postal code, ...",NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,Postal code,Total,Category of citizenship,NaN,Sex,NaN,Five-year age class,NaN,NaN,NaN,...,NaN,NaN,NaN,Marital status 1,NaN,NaN,NaN,NaN,NaN,NaN
2,NaN,NaN,Swiss,Foreigner,Male,Female,0-4,5-9,10-14,15-19,...,80-84,85-89,90 or over,Single,Married,Widowed,Divorced,Unmarried,In a registered partnership,Partnership dissolved
3,Switzerland,9127125,6599008,2528117,4540323,4586802,414051,460540,465467,464482,...,281916,163579,97625,4259310,3624607,402240,825655,755,9500,4346
4,1000,4248,2537,1711,2123,2125,160,232,199,505,...,77,42,16,2583,1335,84,238,0,5,2
5,1003,6879,3523,3356,3559,3320,261,196,189,254,...,89,48,27,4494,1668,157,522,1,25,12
6,1004,31463,17901,13562,15382,16081,1288,1226,1194,1377,...,791,539,369,18628,8588,1169,2987,3,47,34
7,1005,12454,7263,5191,6103,6351,512,477,490,583,...,256,179,132,7671,3340,313,1087,2,20,18
8,1006,15621,9493,6128,7498,8123,652,674,687,690,...,407,239,196,9106,4466,532,1466,1,29,20
9,1007,22716,12530,10186,11042,11674,1048,1109,1036,1177,...,516,288,192,13044,6680,735,2186,3,48,16


In [5]:
# Load only postal code and total population
population_df = pd.read_excel(
    population_file,
    sheet_name="2025",
    usecols="A:B",
    skiprows=3,
    names=["postal_code", "population"]
)

print(population_df.shape)

population_df.head()

(6358, 2)


c:\Users\ichay\anaconda3\Lib\site-packages\openpyxl\worksheet\header_footer.py:48: UserWarning: Cannot parse header or footer so it will be ignored
  warn("""Cannot parse header or footer so it will be ignored""")


,postal_code,population
0,1000,4248.0
1,1003,6879.0
2,1004,31463.0
3,1005,12454.0
4,1006,15621.0


In [6]:
# check data
print("Shape:", population_df.shape)
print()
print(population_df.dtypes)
print()
print(population_df.isna().sum())

Shape: (6358, 2)

postal_code     object
population     float64
dtype: object

postal_code    2
population     5
dtype: int64


In [7]:
population_df[
    population_df["postal_code"].isna()
    | population_df["population"].isna()
]

,postal_code,population
3176,NaN,9127125.0
6353,1 Break in time series: From 2014 onwards excl...,NaN
6354,Source: STATPOP,NaN
6355,© FSO,NaN
6356,NaN,NaN
6357,"Information: Federal Statistical Office (FSO),...",NaN


In [8]:
# Check duplicate postal codes:
print(
    "Duplicate postal codes:",
    population_df["postal_code"].duplicated().sum()
)

Duplicate postal codes: 3177


In [9]:
# Check whether duplicated postcodes have conflicting values
postcode_value_counts = (
    population_df
    .dropna(subset=["postal_code", "population"])
    .groupby("postal_code")["population"]
    .nunique()
)

conflicting_postcodes = postcode_value_counts[
    postcode_value_counts > 1
]

print(
    "Postcodes with different population values:",
    len(conflicting_postcodes)
)

conflicting_postcodes.head()

Postcodes with different population values: 0


Series([], Name: population, dtype: int64)

In [10]:
# Remove missing and exact duplicate rows
population_df = (
    population_df
    .dropna(subset=["postal_code", "population"])
    .drop_duplicates(
        subset=["postal_code", "population"]
    )
    .reset_index(drop=True)
)
print("Shape after cleaning:", population_df.shape)

print(
    "Duplicate postcodes:",
    population_df["postal_code"].duplicated().sum()
)

Shape after cleaning: (3176, 2)
Duplicate postcodes: 0


In [11]:
# Correct the data types
population_df["postal_code"] = (
    pd.to_numeric(
        population_df["postal_code"],
        errors="coerce"
    )
    .astype(int)
    .astype(str)
    .str.zfill(4)
)

population_df["population"] = (
    pd.to_numeric(
        population_df["population"],
        errors="coerce"
    )
    .astype(int)
)
population_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3176 entries, 0 to 3175
Data columns (total 2 columns):
 #   Column       Non-Null Count  Dtype 
---  ------       --------------  ----- 
 0   postal_code  3176 non-null   object
 1   population   3176 non-null   int64 
dtypes: int64(1), object(1)
memory usage: 49.8+ KB


In [12]:
# verify the total
population_total = population_df[
    "population"
].sum()

print(
    "Population total:",
    f"{population_total:,}"
)

Population total: 9,127,125


In [13]:
# save the cleaned data
population_df.to_csv(
    "data/switzerland_population_clean.csv",
    index=False
)

print("Cleaned population data saved successfully")

Cleaned population data saved successfully


In [14]:
# final validation
print("Rows:", len(population_df))
print("Unique postcodes:", population_df["postal_code"].nunique())
print("Missing values:")
print(population_df.isna().sum())
print("Duplicate rows:", population_df.duplicated().sum())

population_df.head()

Rows: 3176
Unique postcodes: 3176
Missing values:
postal_code    0
population     0
dtype: int64
Duplicate rows: 0


,postal_code,population
0,1000,4248
1,1003,6879
2,1004,31463
3,1005,12454
4,1006,15621


### Data-quality observation

The original BFS worksheet contained two identical blocks of postcode-level
population records. Exact duplicate postcode–population pairs were removed
before calculating totals. The cleaned records sum to the official Swiss
population total of 9,127,125.